In [58]:
import gymnasium as gym
import minigrid
from minigrid.wrappers import ImgObsWrapper
from anthropic import Anthropic
import numpy as np
from dotenv import load_dotenv
import hashlib
import json
import os

load_dotenv()

True

In [59]:
from typing import Literal
from pydantic import BaseModel

class Subgoal(BaseModel):
    action: Literal["pick_up", "open_door", "go_to_goal"]
    color: Literal["red", "green", "blue", "purple", "yellow", "grey"] | None = None
    object: Literal["key", "ball", "box"] | None = None

class SubgoalList(BaseModel):
    subgoals: list[Subgoal]

class CacheElement(BaseModel):
    hash: str
    mission:str
    environment: str
    subgoals: list[Subgoal]

In [60]:
class SubgoalCache:
    def __init__(self, path: str):
        self.path = path
        self._entries: dict[str, list[Subgoal]] = {}
        self._load()

    def _load(self) -> None:
        try:
            with open(self.path) as f:
                for line in f:
                    if line.strip():
                        rec = CacheElement.model_validate_json(line)
                        self._entries[rec.hash] = rec.subgoals
        except FileNotFoundError:
            pass

    def __contains__(self, key: str) -> bool:
        return key in self._entries

    def get(self, key: str) -> list[Subgoal] | None:
        return self._entries.get(key)

    def add(self, element: CacheElement) -> None:
        if element.hash in self._entries:
            return
        self._entries[element.hash] = element.subgoals
        with open(self.path, "a") as f:
            f.write(element.model_dump_json() + "\n")

    def __str__(self):
        l = []
        for k,v in self._entries.items():
            l.append(f"{k}, {v}")
        return "\n".join(l)

In [61]:
def make_prompt(mission: str, environment: str) -> str:
    return f"Mission: {mission}\nMap:\n{environment}"

def make_key(mission: str, environment: str) -> str:
    return hashlib.sha256(make_prompt(mission, environment).encode()).hexdigest()

In [62]:
OBJ   = {"wall":"W","floor":"F","door":"D","key":"K","ball":"A","box":"B","goal":"G","lava":"V"}
ADIR  = {0:">",1:"V",2:"<",3:"^"}
CCHAR = {"red":"R","green":"G","blue":"B","purple":"P","yellow":"Y","grey":"E"}

def render_map(base):
    g = base.grid
    rows = []
    for j in range(g.height):
        row = []
        for i in range(g.width):
            if (i, j) == tuple(base.agent_pos):
                row.append(ADIR[base.agent_dir] * 2)
            else:
                o = g.get(i, j)
                if o is None:
                    row.append("  ")
                elif o.type == "door":
                    row.append("__" if o.is_open else (("L" if o.is_locked else "D") + CCHAR[o.color]))
                else:
                    row.append(OBJ[o.type] + CCHAR[o.color])
        rows.append("".join(row))
    return "\n".join(rows)

In [63]:
SYSTEM = """You are a subgoal planner for an agent in a MiniGrid gridworld.
You are given the FULL map of the environment and the agent's mission.
Output the ordered list of subgoals the agent must complete to accomplish the mission.

HOW TO READ THE MAP
The map is printed row by row, from the top row down. Each cell is exactly 2 characters:
the first is the OBJECT, the second is its COLOR.

Objects (1st character):
   W = wall
   D = door
   K = key
   A = ball
   B = box
   G = goal
   V = lava
   F = floor
   '  ' (two spaces) = empty floor (walkable)

Colors (2nd character):
   R = red, G = green, B = blue, P = purple, Y = yellow, E = grey

So 'WE' = grey wall, 'GG' = green goal, 'VR' = lava (lava is always red),
'KY' = yellow key, 'AB' = blue ball.

Doors are special:
   L<color>  = a LOCKED door (e.g. 'LY' = locked yellow door)
   D<color>  = a closed but UNLOCKED door (e.g. 'DB' = closed blue door)
   __        = an already OPEN door (its color is not shown, and it needs no action)

The agent is two identical arrows showing the way it faces:
   >> east,  VV south,  << west,  ^^ north
Note: 'VV' (two arrows) is the agent facing south; lava is 'VR' (V plus a color).
The agent's cell shows the agent, not whatever it is standing on.

Coordinates: columns count left->right from 0 (i); rows count top->bottom from 0 (j);
a cell is (i, j). Use coordinates only to reason about what exists and what blocks what.

YOUR TASK
From the mission and the map, decide which subgoals the agent must achieve and in what order.
A LOCKED door ('L<color>') requires first picking up the matching-color key.
A closed unlocked door ('D<color>') only needs to be opened - no key.
An open door ('__') needs no subgoal at all.
To reach a room you must open the door(s) leading into it. Only propose subgoals for objects
that actually appear on the map or are named in the mission.

SUBGOAL FIELDS
Each subgoal has:
  action: one of 'pick_up', 'open_door', 'go_to_goal'
  color:  required for 'pick_up' and 'open_door' (red, green, blue, purple, yellow, grey)
  object: required for 'pick_up' (key, ball, box)
'go_to_goal' takes no color or object. List subgoals in the exact order the agent should perform them."""

In [64]:
def fetch_subgoals(client:Anthropic, model: str, prompt: str) -> list[Subgoal]:
    resp = client.messages.parse(
        model=model,
        max_tokens=1024,
        system=[{
            "type": "text",
            "text": SYSTEM,
            "cache_control": {"type": "ephemeral"},
        }],
        messages=[{"role": "user", "content": prompt}],
        output_format=SubgoalList
    )
    return resp.parsed_output.subgoals
    ###########
    print(resp.usage.cache_read_input_tokens)
    if resp.usage.cache_read_input_tokens > 0:
        print(f"resp.usage.cache_read_input_tokens: {resp.usage.cache_read_input_tokens}")
    ###########
    raw = next(b.input for b in resp.content if b.type == "tool_use")
    return SubgoalList.model_validate(raw).subgoals

In [ ]:
MODEL = "claude-opus-4-8" #"claude-haiku-4-5"
cache = SubgoalCache(f"subgoal_cache_{MODEL}.jsonl")
client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))

env = gym.make("MiniGrid-MultiRoom-N4-S5-v0", render_mode="human")
base = env.unwrapped
base.reset(seed=42)

# make key for current env 
key = make_key(base.mission, render_map(base))
prompt = make_prompt(base.mission, render_map(base))

if key not in cache:
    cache.add(
        CacheElement(
            hash=key, 
            mission=base.mission, 
            environment=render_map(base), 
            subgoals=fetch_subgoals(client, MODEL, prompt)
        )
    )

subgoals = cache.get(key)

print(subgoals)

[Subgoal(action='open_door', color='grey', object=None), Subgoal(action='open_door', color='purple', object=None), Subgoal(action='open_door', color='green', object=None), Subgoal(action='open_door', color='blue', object=None), Subgoal(action='go_to_goal', color=None, object=None)]


: 